# Cross-encoder evidence retriever (FinQANet-style stage 1)

**Research question.** Can a cross-encoder trained on train `gold_inds` retrieve *all* gold facts more
often than train-fitted TF-IDF? Notebook 09 measured TF-IDF all-gold recall@3 at 53.8% on train
(in-sample), and 28.5% for multi-fact records. The generator in Notebook 11 can only use facts that
are retrieved.

**Protocol.** Settings come from `configs/retriever_bert_base.toml`, following the official FinQA
retriever where stated. Train is used for fitting. Dev is used for checkpoint selection and reporting.
The official test split is not loaded. Prediction inputs are built only from `id`, `question`, `table`,
`pre_text` and `post_text`; dev `gold_inds` are read only after scoring, to compute recall.

**Runtime.** This notebook needs a CUDA GPU: follow `docs/gpu-setup.md` first. Setting
`FINEVIDENCE_SMOKE_TEST=1` runs a tiny CPU check on a few records with a tiny random model. Smoke
outputs go to separate paths and are not results.

In [ ]:
from __future__ import annotations

import json
import os
import platform
import random
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

from finevidence.data import load_finqa_split
from finevidence.data.acquisition import download_finqa_splits
from finevidence.evaluation import gold_recall_at_k
from finevidence.retrieval import (
    build_ranking_pairs,
    build_training_pairs,
    load_retriever_config,
    rank_keys,
    sample_epoch_pairs,
)

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
SMOKE_TEST = os.environ.get("FINEVIDENCE_SMOKE_TEST") == "1"
config = load_retriever_config(PROJECT_ROOT / "configs" / "retriever_bert_base.toml")

pretrained_name = (
    "hf-internal-testing/tiny-random-BertModel" if SMOKE_TEST else config.pretrained_name
)
epochs = 1 if SMOKE_TEST else config.epochs
suffix = "-smoke" if SMOKE_TEST else ""
model_dir = Path(str(config.model_dir) + suffix)
model_dir = model_dir if model_dir.is_absolute() else PROJECT_ROOT / model_dir
predictions_path = PROJECT_ROOT / config.predictions_path
if SMOKE_TEST:
    predictions_path = predictions_path.with_name(predictions_path.stem + "_smoke.jsonl")

random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
torch.cuda.manual_seed_all(config.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not SMOKE_TEST and device.type != "cuda":
    raise RuntimeError(
        "Full training needs a CUDA GPU; set FINEVIDENCE_SMOKE_TEST=1 for a CPU check."
    )

commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=PROJECT_ROOT
).stdout.strip()
environment = {
    "commit": commit,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
    "smoke_test": SMOKE_TEST,
    "pretrained_name": pretrained_name,
    "epochs": epochs,
    "seed": config.seed,
}
print(json.dumps(environment, indent=2))

## 1. Verified data and retriever pairs

The acquisition helper downloads train and dev only if they are missing, and verifies both against the
pinned SHA-256 values. Positives and negatives use the same header-paired rendering of the raw context
(see `finevidence.retrieval.pairs`).

In [ ]:
raw_dir = PROJECT_ROOT / config.dataset_root
print(download_finqa_splits(raw_dir, (config.train_split, config.evaluation_split)))
train = load_finqa_split(raw_dir / f"{config.train_split}.json")
dev = load_finqa_split(raw_dir / f"{config.evaluation_split}.json")
if SMOKE_TEST:
    train, dev = train[:40], dev[:20]

positives, negative_pool, unmatched_gold = build_training_pairs(
    train, include_header_row=config.include_header_row
)
dev_pairs = {
    record["id"]: build_ranking_pairs(record, include_header_row=config.include_header_row)
    for record in dev
}
print(
    {
        "train_records": len(train),
        "dev_records": len(dev),
        "test_loaded": False,
        "train_positives": len(positives),
        "train_negative_pool": len(negative_pool),
        "pairs_per_epoch": len(positives) * (1 + config.negatives_per_positive),
        "dev_pairs": sum(len(pairs) for pairs in dev_pairs.values()),
        "unmatched_train_gold_keys": dict(unmatched_gold),
    }
)
display(
    pd.DataFrame([vars(pair) for pair in (*positives[:3], *negative_pool[:3])])[
        ["record_id", "key", "label", "question", "text"]
    ]
)

## 2. TF-IDF reference on dev

This is the lexical reference on the same rendering. The vocabulary is fitted on train questions and
train segment texts only. Dev gold keys are read only after the rankings are fixed.

In [ ]:
def recall_table(rankings: dict[str, list[str]], records: list[dict]) -> pd.DataFrame:
    rows = []
    for record in records:
        gold = [key for key in record["qa"]["gold_inds"] if key in rankings[record["id"]]]
        if not gold:
            continue
        for k in config.recall_k:
            recall = gold_recall_at_k(rankings[record["id"]], gold, k)
            rows.append(
                {
                    "k": k,
                    "any_gold": recall.any_gold,
                    "all_gold": recall.all_gold,
                    "multi_fact": recall.gold_count > 1,
                }
            )
    frame = pd.DataFrame(rows)
    table = frame.groupby("k")[["any_gold", "all_gold"]].mean()
    table["all_gold_multi_fact"] = frame[frame.multi_fact].groupby("k")["all_gold"].mean()
    return table.round(4)


vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=50000, sublinear_tf=True).fit(
    [pair.text for pair in (*positives, *negative_pool)]
    + [record["qa"]["question"] for record in train]
)
tfidf_rankings = {}
for record_id, pairs in dev_pairs.items():
    scores = (
        vectorizer.transform([p.text for p in pairs]) @ vectorizer.transform([pairs[0].question]).T
    )
    tfidf_rankings[record_id] = rank_keys([p.key for p in pairs], scores.toarray().ravel().tolist())
tfidf_recall = recall_table(tfidf_rankings, dev)
display(tfidf_recall)

## 3. Train the cross-encoder

`AutoModelForSequenceClassification` with two labels scores `(question, segment)` pairs; only the
segment side is truncated. Each epoch resamples negatives with a seed derived from the config seed
and the epoch. The checkpoint with the best dev all-gold recall at `top_n_for_generator` is kept.
Mixed precision is used on CUDA.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(pretrained_name)
model = AutoModelForSequenceClassification.from_pretrained(pretrained_name, num_labels=2).to(device)


def encode(pairs):
    return tokenizer(
        [pair.question for pair in pairs],
        [pair.text for pair in pairs],
        truncation="only_second",
        max_length=config.max_seq_length,
        padding=True,
        return_tensors="pt",
    ).to(device)


@torch.no_grad()
def rank_dev() -> tuple[dict[str, list[str]], dict[str, list[float]]]:
    model.eval()
    rankings, scores_by_record = {}, {}
    for record_id, pairs in dev_pairs.items():
        scores = []
        for start in range(0, len(pairs), config.batch_size * 4):
            batch = pairs[start : start + config.batch_size * 4]
            with torch.autocast(device.type, enabled=device.type == "cuda"):
                logits = model(**encode(batch)).logits
            scores.extend(torch.softmax(logits.float(), dim=-1)[:, 1].tolist())
        rankings[record_id] = rank_keys([p.key for p in pairs], scores)
        scores_by_record[record_id] = scores
    return rankings, scores_by_record


steps_per_epoch = -(-len(positives) * (1 + config.negatives_per_positive) // config.batch_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(config.warmup_ratio * steps_per_epoch * epochs), steps_per_epoch * epochs
)
scaler = torch.amp.GradScaler(enabled=device.type == "cuda")
selection_k = config.top_n_for_generator
history, best_recall = [], -1.0
model_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(epochs):
    model.train()
    epoch_pairs = sample_epoch_pairs(
        positives,
        negative_pool,
        negatives_per_positive=config.negatives_per_positive,
        seed=config.seed,
        epoch=epoch,
    )
    started, losses = time.time(), []
    for start in range(0, len(epoch_pairs), config.batch_size):
        batch = epoch_pairs[start : start + config.batch_size]
        labels = torch.tensor([pair.label for pair in batch], device=device)
        with torch.autocast(device.type, enabled=device.type == "cuda"):
            loss = model(**encode(batch), labels=labels).loss
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        losses.append(loss.item())
    rankings, _ = rank_dev()
    recall = recall_table(rankings, dev)
    selected = float(recall.loc[selection_k, "all_gold"])
    history.append(
        {
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "train_seconds": round(time.time() - started, 1),
            f"dev_all_gold_at_{selection_k}": selected,
            f"dev_any_gold_at_{selection_k}": float(recall.loc[selection_k, "any_gold"]),
        }
    )
    print(history[-1])
    if selected > best_recall:
        best_recall = selected
        model.save_pretrained(model_dir / "best")
        tokenizer.save_pretrained(model_dir / "best")
(model_dir / "history.json").write_text(
    json.dumps({"environment": environment, "history": history}, indent=2)
)
display(pd.DataFrame(history))

## 4. Selected checkpoint on dev and ranking artifact

The selected checkpoint re-ranks every dev record. The full ranking with scores is written to an ignored
path for Notebook 11. The cross-encoder and TF-IDF are then compared on the same dev records and rendering.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_dir / "best").to(device)
rankings, scores_by_record = rank_dev()
cross_encoder_recall = recall_table(rankings, dev)
comparison = pd.concat({"tfidf": tfidf_recall, "cross_encoder": cross_encoder_recall}, axis=1)
display(comparison)

predictions_path.parent.mkdir(parents=True, exist_ok=True)
with predictions_path.open("w", encoding="utf-8") as handle:
    for record_id, pairs in dev_pairs.items():
        score_by_key = dict(zip([p.key for p in pairs], scores_by_record[record_id], strict=True))
        ranked = rankings[record_id]
        handle.write(
            json.dumps(
                {
                    "id": record_id,
                    "ranked_keys": ranked,
                    "scores": [round(score_by_key[key], 6) for key in ranked],
                    "checkpoint": str(model_dir / "best"),
                    "commit": commit,
                }
            )
            + "\n"
        )
print(
    {"rankings": str(predictions_path), "records": len(rankings), "best_epoch_metric": best_recall}
)

## Interpretation

Write this section only after a full (non-smoke) GPU run, from the printed outputs above:
the selected epoch, the dev recall table against TF-IDF, multi-fact all-gold recall, training time,
GPU type, and any limitation. Then add the experiment-log entry. Smoke-test outputs are not results.